# 从零开始手搓 Diffusion Model 之 DDPM


1. `TimeEmbedding`：把扩散步数 `t` 编码成网络能用的向量。
2. `ResBlock` / `AttnBlock`：构成带时间条件的 UNet 基础模块。
3. `UNet`：输入带噪图片 `x_t` 和时间步 `t`，预测噪声 `epsilon_theta(x_t, t)`。
4. `GaussianDiffusionTrainer`：训练阶段的前向加噪和 MSE 噪声预测损失。
5. `GaussianDiffusionSampler`：推理阶段从纯噪声一步步反推生成图片。

默认以 MNIST 的 1 通道 28x28 图片为例，图片数值范围使用 `[-1, 1]`。

In [1]:
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init


def get_device():
    """选择运行设备：优先 CUDA；如果 CUDA 初始化失败，就自动回退到 CPU。"""
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device='cuda') + 1
            return torch.device('cuda')
        except RuntimeError as err:
            print(f'CUDA 当前不可用，自动切换到 CPU：{err}')
    return torch.device('cpu')


def extract(v, t, x_shape):
    """
    从长度为 T 的一维系数表 v 中，按 batch 内每张图片的时间步 t 取出对应系数。

    参数：
    - v: shape [T]，例如 sqrt_alphas_bar、betas 等预先算好的扩散系数。
    - t: shape [B]，每张图片各自采样到的时间步。
    - x_shape: x 的完整形状，例如 [B, C, H, W]。

    返回：
    - shape [B, 1, 1, 1]，这样可以和图片张量 [B, C, H, W] 自动广播相乘。
    """
    out = v.gather(dim=0, index=t)
    return out.float().view(t.shape[0], *((1,) * (len(x_shape) - 1)))


class Swish(nn.Module):
    """Swish 激活函数：x * sigmoid(x)，DDPM / UNet 里很常见。"""
    def forward(self, x):
        return x * torch.sigmoid(x)


def group_norm(channels):
    """
    GroupNorm 要求 channels 能被 num_groups 整除。
    经典代码常写死 32 组；这里做一个小封装，避免通道数较小时出错。
    """
    groups = min(32, channels)
    while channels % groups != 0:
        groups -= 1
    return nn.GroupNorm(groups, channels)


device = get_device()
print('当前设备：', device)





当前设备： cuda


In [ ]:
#正式手搓ddpm

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def extract(v, t, x_shape):
    out = v.gather(dim =0, index=t)
    return out.float().view(t.shape[0],*((1,)*(len(x_shape)-1)))#这个return的向量形状不是很清楚，为什么要这么变形？因为这个函数的作用是从长度为 T 的一维系数表 v 中，按 batch 内每张图片的时间步 t 取出对应系数。返回的向量形状是 [B, 1, 1, 1]，这样可以和图片张量 [B, C, H, W] 自动广播相乘。

class Swish(nn.Module):
     def forward(self,x):
          return x*torch.sigmoid(x)


class TimeEmbedding(nn.Module):
    def __init__(self,T,d_model,dim):
        super().__init__()
        emb = torch.arange(0, d_model, step = 2).float()/d_model * math.log(10000)
        emb = torch.exp(-emb)

        pos = torch.arange(T).float().unsqueeze(1)
        #这个的意思是加维度 
        emb = emb.unsqueeze(0)
        emb = pos * emb #为什么要乘起来? 
        emb = torch.stack([torch.sin(emb),torch.cos(emb)],dim=-1)
        emb = emb.view(T, d_model)#这向量是怎么个变化？ 
        self.timeembedding = nn.Sequential(
            nn.Embedding.from_pretrained(emb,freeze=False),#这个的意思是把上面算好的表作为初始权重，freeze=False表示在训练过程中这个权重是可以更新的。
            nn.Linear(d_model,dim),
            nn.SiLU(),
            nn.Linear(dim,dim)
        )
        self.initialize()

    def initialize(self):
            for _ in self.modules():
                if isinstance(_,nn.Linear):
                    init.xavier_uniform_(_.weight)
                    init.zeros_(_.bias)
    def forward(self,t):
        return self.timeembedding(t)
    

'''注意力机制（卷积版）'''
class AttnBlock(nn.Module):
    def __init__(self,in_ch):
        super().__init__()
        self.gropnorm = group_norm(in_ch)

        self.proj_q = nn.Conv2d(in_ch,in_ch,kernel_size=1,stride=1,padding=0)
        self.proj_k = nn.Conv2d(in_ch,in_ch,kernel_size=1,stride=1,padding=0)
        self.proj_v = nn.Conv2d(in_ch,in_ch,kernel_size=1,stride=1,padding=0)
        self.proj = nn.Conv2d(in_ch,in_ch,kernel_size=1,stride=1,padding=0)
        self.initialize()

    def initialize(self):
        for module in [self.proj_q,self.proj_k,self.proj_v,self.proj]:
             init.xavier_normal(module.weight)
             init.zeros_(module.bias)

             init.xavier_normal(self.proj.weight,gain = 1e-5)

    def forward(self,x):
         B,C,H,W = x.shape
         h = self.gropnorm(x)

         q = self.proj_q(h)
         k = self.proj_k(h)
         v = self.proj_v(h)

         q = q.permute(0,2,3,1).view(B,H*W,C)
         k = k.view(B,C,H*W)

         w = torch.matmul(q,k)*(int(C)**-0.5)

         w =  F.softmax(w,dim = -1) #dim=-1的含义?是指在最后一个维度上进行softmax操作，也就是对每个像素位置的注意力权重进行归一化，使得它们的和为1。

         v = v.permute(0,2,3,1).view(B,H*W,C)

         score = torch.matmul(w,v)
         score = score.view(B,H,W,C).permute(0,3,1,2)
         score = self.proj(score)

         return score+x
    

#ResBlock结构：
class Resblock(nn.Module):
    def __init__(self,in_ch,out_ch,tdim,dropout,attn = False):
        super().__init__()

        self.block1 = nn.Sequential(
            group_norm(in_ch),
            Swish(),
            nn.Conv2d(in_ch,out_ch,kernel_size=3,stride=1,padding=1),#为什么要用3*3的卷积核？
        )
          
        self.tdim_proj = nn.Sequential(
            Swish(),
            nn.Linear(tdim,out_ch),
        )

        self.block2 = nn.Sequential(
            group_norm(out_ch),
            Swish(),
            nn.Dropout(dropout),
            nn.Conv2d(out_ch,out_ch,3,stride=1,padding=1),
        )
        

        if in_ch != out_ch:
            self.shortcut = nn.Linear(in_ch,out_ch)
        else :
            self.shortcut = nn.Identity()

        self.attn = AttnBlock(out_ch) if attn else nn.Identity()
        self.initialize()

    def initialize(self):
        for _ in self.modules():
            if isinstance(_,(nn.Conv2d,nn.Linear)):
                init.xavier_normal_(_.weight)
                init.zeros_(_.bias)

    def forward(self,x,temb):#为什么这个temb有两维呢？因为这个temb是时间嵌入的结果，通常是一个二维张量，形状为 [B, tdim]，其中 B 是批量大小，tdim 是时间嵌入的维度。这个时间嵌入向量会被映射成与图像特征图相同的通道数，并通过广播机制加到特征图上，以便在每个时间步都能提供时间信息。  
        h = self.block1(x)
        h = h + self.tdim_proj(temb)[:,:,None,None] 
        h = self.block2(h)
        h = h + self.shortcut(h)

        h = self.attn(h)

        return h
    
#下采样和上采样的过程class DownSample(nn.Module):
class DownSample(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.main = nn.Conv2d(ch,ch,kernel_size= 3 ,stride= 2 ,padding=1)
        init.xavier_normal(self.main.weight)
        init.zeros_(self.main.bias)

    def forward(self,x):
        x = self.main(x)
        return x 
    
class UpSample (nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.main = nn.Conv2d(ch,ch,3,stride=1,padding=1)
        init.xavier_normal(self.main)
        init.zeros_(self.main.bias)

    def forward(self,x):
        x = F.interpolate(x,scale_factor=2,mode='nearest')
        return self.main(x)
        



#UNet
class UNet(nn.Module):
    def __init__(self,
        T,
        ch = 64,
        ch_mult=(1,2,2),#这是干什么的？这是每个分辨率层级的通道倍率；(1, 2, 2) 表示 64 -> 128 -> 128。
        attn= (1,),#这个是干什么的？为什么加逗号？这是一个元组，表示在哪些层级使用注意力机制；这里默认第 1 层级，也就是 14x14 附近。加逗号是为了让 Python 识别这是一个单元素的元组，而不是一个普通的括号表达式。
        num_res_blocks=2,
        dropout = 0.1,
        in_ch=1,
        out_ch=1,
        ):

        super().__init__()
        self.T = T
        self.ch=ch
        self.tdim=ch*4#为什么*4？这是时间嵌入的维度，通常设置为基础通道数的4倍，以提供足够的表达能力。
        self.num_res_blocks = num_res_blocks

        self.time_embedding=TimeEmbedding(T,ch,self.tdim)
        self.head = nn.Conv2d(in_ch,ch,3,stride=1,padding=1)
#Encoder部分
        self.downblocks = nn.ModuleList()
        chs = [ch]
        now_ch = ch
        for i,mult in enumerate(ch_mult):
            out_channels = ch*mult
            for _ in range(num_res_blocks):
                self.downblocks.append(ResBlock(now_ch,out_channels,self.tdim,dropout,attn=(i in attn)))
                now_ch = out_channels
                chs.append(now_ch)

            if i != len(ch_mult)-1:
                self.downblocks.append(DownSample(now_ch))
                chs.append(now_ch)


#Middle:
        self.middleblocks = nn.Module([
            ResBlock(now_ch,now_ch,self.tdim,dropout,attn=True),
            ResBlock(now_ch,now_ch,self.tdim,dropout,attn=False),
        ])
        

#Decoder:上采样过程
        self.upblocks = nn.ModuleList()
        for i,mult in reversed(list(enumerate(ch_mult))):
            out_channels=ch*mult
            for _ in range(num_res_blocks+1):
                skip_ch = chs.pop()#这是干什么？chs.pop()是什么用法？这是从列表 chs 中弹出最后一个元素，表示当前层级对应的 skip connection 的通道数。这个通道数会和当前特征图的通道数相加，作为 ResBlock 的输入通道数。
                self.upblocks.append(ResBlock(now_ch+skip_ch,out_channels,self.tdim,dropout,attn=(i in attn)))
                now_ch = out_channels

            if i != 0:
                self.upblocks.append(UpSample(now_ch))

            self.tail = nn.Sequential(
                group_norm(now_ch),
                Swish(),
                nn.Conv2d(now_ch,out_ch,3,stride=1,padding=1),
            )
            self.initialize()

        def initialize(self):
            init.xavier_normal(self.head.weight)
            init.zeros_(self.head.bias)
            init.xavier_normal(self.tail[-1].weight,gain=1e-5)
            init.zeros_(self.tail[-1].bias)
            
        def forward(self,x,t):
            temb = self.time_embbedding(t)

            h = self.head(x)
            hs = [h]

            for layer in self.downblocks:
                if isinstance (layer,ResBlock):
                    h = layer(h,temb)
                else :
                    h = layer(h)
                hs.append(h)

            for layer in self.middleblocks:
                h = layer(h,temb)#为什么这个layer只要俩个参数？因为 middleblocks 中的 ResBlock 只需要图像特征和时间嵌入作为输入，不涉及下采样或上采样操作，所以不需要额外的参数.
            #为什么可以把h整个传进去？因为 ResBlock 的 forward 方法定义为 def forward(self, x, temb)，其中 x 是图像特征，temb 是时间嵌入。无论是在 encoder 还是 middle 部分，输入到 ResBlock 的都是当前的图像特征 h 和时间嵌入 temb，所以直接把 h 传进去就可以了。
            #这个layer是什么东西？其他的layer也是resblock吗？这个layer是 middleblocks 中的 ResBlock 实例，其他的 layer 可能是 downblocks 中的 ResBlock 或 DownSample，upblocks 中的 ResBlock 或 UpSample。根据 isinstance 的判断，代码会自动区分不同类型的 layer，并传入相应的参数。
            

            #Decoder :
            for layer in self.upblocks:
                if isinstance(layer,ResBlock):
                    skip = hs.pop
                    if h.shape[-2:] != skip.shape[-2:]:#h的形状是什么？skip的形状是什么？h 的形状是当前的图像特征图，通常是 [B, C, H, W]；skip 的形状是对应的 skip connection 特征图，通常也是 [B, C_skip, H_skip, W_skip]。如果它们的空间尺寸不一致，就需要通过插值对齐。
                        h = F.interpolate(h,size=skip.shape[-2:],mode='nearest')

                        h = torch.cat([h,skip],dim=1)
                        h = layer(h,temb)
                    else :
                        h = layer(h)

                return self.tail(h)
            

#DDPM训练器
class GaussianDiffusionTrainer(nn.Module):
    def __init__(self,model,bata_1 = 1e-4,beta_T = 0.02,T=1000):
        super().__init__()
        self.model = model
        self.T = T
        self.register_buffer('betas',torch.linspace(bata_1,beta_T,T).float())#这个的意思是创建一个长度为 T 的一维张量，线性地从 beta_1 增加到 beta_T，并注册为模型的 buffer，这样它就会随着模型一起保存和加载，但不会被优化器更新。
        #为什么beta_T这么小？因为 beta_T 是扩散过程最后一步加入的噪声强度，设置得太大可能会导致生成的图像质量下降；设置得太小可能会导致训练不稳定。通常在 0.01 到 0.02 之间是比较常见的选择。
        alphas = 1.0- self.betas 
        alphas_bar = torch.cumprod(alphas,dim=0)#这个函数的作用是计算 alphas 的累积乘积，得到 alpha_bar_t = alpha_1 * alpha_2 * ... * alpha_t，这个值在训练公式中用于计算 x_t 和噪声的权重。
        self.register_buffer('sqrt_alphas_bar',torch.sqrt(alphas_bar))
        self.register_buffer('sqrt_one_minus_alphas_bar',torch.sqrt(1.0-alphas_bar))#这一步是？这是为了在训练公式中直接使用 sqrt(alpha_bar_t) 和 sqrt(1 - alpha_bar_t)，避免每次计算时都要进行平方根运算，提高效率。


        def forward(self,x_0):
            B = x_0.shape[0]

            t = torch.randint(self.T,size = (B,),device=x_0.device)#这个函数的写法是这样的吗？每个参数的意义是什么？这是 PyTorch 中的一个函数，用于生成一个形状为 (B,) 的整数张量，元素值在 [0, T) 的范围内，表示每张图片随机抽取的时间步。size 参数指定输出张量的形状，device 参数指定输出张量所在的设备（CPU 或 GPU）。

            noise = torch.randn_like(x_0)
            #这是什么函数？这是 PyTorch 中的一个函数，用于生成与 x_0 形状相同的张量，元素值服从标准正态分布（均值为 0，标准差为 1）。这个噪声张量是用来模拟扩散过程中的随机噪声的。

            x_t = (
                extract(self.sqrt_alphs_bar,t ,x_0.shape)*x_0+
                extract(self.sqrt_one_minus_alphas_bar,t,x_0.shape)*noise
            )

            pred_noise = self.model(x_t,t)
            loss = F.mse_loss(pred_noise,noise,reduction="mean")
            return loss
        

#DDPM采样：
class GaussianDiffusionSampler(nn.Module):
    def __init__(self, model, beta_1=1e-4, beta_T=0.02, T=1000):
        super().__init__()
        self.model = model
        self.T = T

        self.register_buffer('betas', torch.linspace(beta_1, beta_T, T).float())
        alphas = 1.0 - self.betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        alphas_bar_prev = F.pad(alphas_bar,[1,0],value=1.0)[:T]
        #这里面每个参数是什么意思？这是为了计算反向分布的方差时需要用到的 alpha_bar_{t-1}，由于 alpha_bar 的第 0 项对应 t=0 时的值为 1，所以通过在前面填充一个 1.0 来实现对齐。pad 参数 [1, 0] 表示在第一个维度前面填充 1 个元素，后面不填充；value=1.0 表示填充的值为 1.0；[:T] 是为了去掉多余的最后一项，使得 alphas_bar_prev 的长度与 T 一致。
        

        self.register_buffer('coeff1',torch.sqrt(1.0/alphas))
        self.register_buffer('coeff2',self.coeff1*(1.0 - alphas)/torch.sqrt(1.0-alphas_bar))
        self.register_buffer('posterior_var', self.betas * (1.0 - alphas_bar_prev) / (1.0 - alphas_bar))

    def predict_xt_prev_mean_from_eps(self,x_t,t,eps):
        return (
            extract(self.coeff1,t,x_t.shape)*x_t-
            extract(self.corff2,t,x_t.shape)*eps
        )
    
    def p_mean_varince(self,x_t,t):
        var = torch.cat([self.posteriior_var[1:2],self.betas[1:]])#这一行什么意思？这是为了处理 t=0 时 posterior_var 为 0 的情况，避免数值不稳定。通过将 posterior_var 的第 1 项（对应 t=1 的值）替代第 0 项，使得在 t=0 时也有一个合理的方差值，从而保证采样过程的稳定性。
        var = extract(var,t,x_t.shape)

        eps = self.model(x_t,t)
        xt_prev_mean = self.predict_xt_prev_mean_from_eps(x_t,t,eps=eps)
        return xt_prev_mean,var
    
    @torch.no_grad()
    def forward(self,x_T):
        x_t = x_T
        for time_step in reversed(range(self.T)):
            t = x_t.new_full((x_t.shape[0],),time_step,dtype = torch.long)
            mean,var = self.p_mean_varince(x_t=x_t,t=t)

            if time_step > 0:
                noise = torch.randn_like(x_t)

            else :
                noise = 0

            x_t = mean + torch.sqrt(var)*noise

        x_0 = x_t
        return torch.clip(x_0,-1.0,1.0)#clip在干什么？这是为了确保生成的图像像素值在 [-1, 1] 的范围内，因为在训练阶段我们将输入图像归一化到了这个范围。clip 函数会将 x_0 中的值限制在指定的范围内，超过 -1.0 的部分会被设置为 -1.0，超过 1.0 的部分会被设置为 1.0，从而保证输出的图像数据合法.





## 1. 时间步嵌入 TimeEmbedding

DDPM 的去噪网络不是只看图片 `x_t`，还必须知道当前是第几个扩散步 `t`。这里先用 sin/cos 位置编码构造一个 `[T, d_model]` 的表，再经过两层 MLP 变成残差块需要的时间条件向量。

In [3]:
class TimeEmbedding(nn.Module):
    def __init__(self, T, d_model, dim):
        """
        参数：
        - T: 总扩散步数，例如 1000。
        - d_model: 原始 sin/cos 时间编码维度，必须是偶数。
        - dim: 最终输出维度，通常等于 UNet 的时间嵌入维度 tdim。
        """
        assert d_model % 2 == 0, 'd_model 必须是偶数，因为 sin/cos 要成对出现'
        super().__init__()

        # emb: [d_model / 2]
        # 这一项对应 Transformer 位置编码里的不同频率。
        emb = torch.arange(0, d_model, step=2).float() / d_model * math.log(10000)
        emb = torch.exp(-emb)

        # pos: [T]，表示 t = 0, 1, ..., T-1。
        pos = torch.arange(T).float()

        # [T, 1] * [1, d_model/2] -> [T, d_model/2]
        emb = pos[:, None] * emb[None, :]
        assert list(emb.shape) == [T, d_model // 2]

        # 把每个频率分别做 sin 和 cos，然后拼回 d_model 维。
        emb = torch.stack([torch.sin(emb), torch.cos(emb)], dim=-1)
        assert list(emb.shape) == [T, d_model // 2, 2]
        emb = emb.view(T, d_model)

        self.timeembedding = nn.Sequential(
            # Embedding.from_pretrained 会把上面算好的表作为初始权重。
            nn.Embedding.from_pretrained(emb, freeze=False),
            nn.Linear(d_model, dim),
            Swish(),
            nn.Linear(dim, dim),
        )
        self.initialize()

    def initialize(self):
        # 只初始化 Linear；Embedding 的 sin/cos 初始值保留。
        for module in self.modules():
            if isinstance(module, nn.Linear):
                init.xavier_uniform_(module.weight)
                init.zeros_(module.bias)

    def forward(self, t):
        # t: [B]，每张图片对应一个扩散步；返回 [B, dim]。
        return self.timeembedding(t)


## 2. 注意力块 AttnBlock

卷积擅长看局部，注意力可以让一个位置直接关注整张图的其它位置。这里使用最朴素的单头 self-attention，适合放在较小分辨率特征图上，例如 7x7 或 14x14。

In [ ]:
class AttnBlock(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.group_norm = group_norm(in_ch)

        # 用 1x1 卷积生成 q/k/v；空间尺寸不变，只改变每个像素位置的通道表示。
        self.proj_q = nn.Conv2d(in_ch, in_ch, 1, stride=1, padding=0)
        self.proj_k = nn.Conv2d(in_ch, in_ch, 1, stride=1, padding=0)
        self.proj_v = nn.Conv2d(in_ch, in_ch, 1, stride=1, padding=0)
        self.proj = nn.Conv2d(in_ch, in_ch, 1, stride=1, padding=0)
        self.initialize()

    def initialize(self):
        for module in [self.proj_q, self.proj_k, self.proj_v, self.proj]:
            init.xavier_uniform_(module.weight)
            init.zeros_(module.bias)

        # 最后一层用很小的初始化，让注意力分支一开始接近 0，训练更稳定。
        init.xavier_uniform_(self.proj.weight, gain=1e-5)

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        h = self.group_norm(x)

        q = self.proj_q(h)
        k = self.proj_k(h)
        v = self.proj_v(h)

        # q: [B, C, H, W] -> [B, H*W, C]
        # k: [B, C, H, W] -> [B, C, H*W]
        q = q.permute(0, 2, 3, 1).view(B, H * W, C)
        k = k.view(B, C, H * W)

        # 注意力权重 w: [B, H*W, H*W]，表示每个像素位置对其它位置的关注程度。
        w = torch.bmm(q, k) * (int(C) ** -0.5)
        assert list(w.shape) == [B, H * W, H * W]
        w = F.softmax(w, dim=-1)

        # v: [B, C, H, W] -> [B, H*W, C]
        v = v.permute(0, 2, 3, 1).view(B, H * W, C)

        # 根据注意力权重汇聚 value，再还原成图片特征图形状。
        h = torch.bmm(w, v)
        assert list(h.shape) == [B, H * W, C]
        h = h.view(B, H, W, C).permute(0, 3, 1, 2)
        h = self.proj(h)

        # 残差连接：注意力只负责补充信息，不直接替换原特征。
        return x + h


## 3. 残差块、上下采样和 UNet

`ResBlock` 做三件事：

1. 对图片特征 `x` 做 `GroupNorm + Swish + Conv`。
2. 把时间嵌入 `temb` 通过 MLP 投影到通道维，然后加到特征图上。
3. 再做一次 `GroupNorm + Swish + Dropout + Conv`，最后加上 shortcut。

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, dropout, attn=False):
        super().__init__()

        # 第一条分支处理图像特征 x。
        self.block1 = nn.Sequential(
            group_norm(in_ch),
            Swish(),
            nn.Conv2d(in_ch, out_ch, 3, stride=1, padding=1),
        )

        # 第二条分支把时间嵌入 temb 映射成 out_ch 维，再广播加到特征图上。
        self.temb_proj = nn.Sequential(
            Swish(),
            nn.Linear(tdim, out_ch),
        )

        # 第三条分支继续卷积融合。
        self.block2 = nn.Sequential(
            group_norm(out_ch),
            Swish(),
            nn.Dropout(dropout),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1),
        )

        # 如果输入输出通道不同，用 1x1 卷积把 shortcut 的通道对齐。
        if in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride=1, padding=0)
        else:
            self.shortcut = nn.Identity()

        self.attn = AttnBlock(out_ch) if attn else nn.Identity()
        self.initialize()

    def initialize(self):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    init.zeros_(module.bias)

        # block2 的最后一层接近 0 初始化，让残差块初始更像 identity，训练更稳。
        init.xavier_uniform_(self.block2[-1].weight, gain=1e-5)

    def forward(self, x, temb):
        h = self.block1(x)

        # temb_proj(temb): [B, out_ch]
        # [:, :, None, None] -> [B, out_ch, 1, 1]，从而能加到 [B, out_ch, H, W] 上。
        h = h + self.temb_proj(temb)[:, :, None, None]
        h = self.block2(h)

        h = h + self.shortcut(x)
        h = self.attn(h)
        return h


class DownSample(nn.Module):
    """下采样：宽高减半，通道数不变。28x28 -> 14x14 -> 7x7。"""
    def __init__(self, ch):
        super().__init__()
        self.main = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
        init.xavier_uniform_(self.main.weight)
        init.zeros_(self.main.bias)

    def forward(self, x):
        return self.main(x)


class UpSample(nn.Module):
    """上采样：先最近邻放大 2 倍，再用卷积融合，避免转置卷积棋盘格问题。"""
    def __init__(self, ch):
        super().__init__()
        self.main = nn.Conv2d(ch, ch, 3, stride=1, padding=1)
        init.xavier_uniform_(self.main.weight)
        init.zeros_(self.main.bias)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        return self.main(x)


In [ ]:
class UNet(nn.Module):
    def __init__(
        self,
        T,
        ch=64,
        ch_mult=(1, 2, 2),
        attn=(1,),
        num_res_blocks=2,
        dropout=0.1,
        in_ch=1,
        out_ch=1,
    ):
        """
        一个适合 MNIST 28x28 的轻量 UNet。

        参数：
        - T: 扩散步数，传给 TimeEmbedding。
        - ch: 基础通道数。
        - ch_mult: 每个分辨率层级的通道倍率；(1, 2, 2) 表示 64 -> 128 -> 128。
        - attn: 哪些层级使用 attention；这里默认第 1 层级，也就是 14x14 附近。
        - num_res_blocks: 每个分辨率层级堆几个 ResBlock。
        - in_ch/out_ch: MNIST 是 1 通道；RGB 图片可以改成 3。
        """
        super().__init__()
        self.T = T
        self.ch = ch
        self.tdim = ch * 4
        self.num_res_blocks = num_res_blocks

        self.time_embedding = TimeEmbedding(T, ch, self.tdim)
        self.head = nn.Conv2d(in_ch, ch, 3, stride=1, padding=1)

        # ---------- Encoder：逐级下采样，同时保存 skip connection ----------
        self.downblocks = nn.ModuleList()
        chs = [ch]
        now_ch = ch
        for i, mult in enumerate(ch_mult):
            out_channels = ch * mult
            for _ in range(num_res_blocks):
                self.downblocks.append(ResBlock(now_ch, out_channels, self.tdim, dropout, attn=(i in attn)))
                now_ch = out_channels
                chs.append(now_ch)

            # 最后一层不再下采样，否则 MNIST 的 7x7 会继续变得太小。
            if i != len(ch_mult) - 1:
                self.downblocks.append(DownSample(now_ch))
                chs.append(now_ch)

        # ---------- Middle：UNet 最底部，先 ResBlock+Attn，再 ResBlock ----------
        self.middleblocks = nn.ModuleList([
            ResBlock(now_ch, now_ch, self.tdim, dropout, attn=True),
            ResBlock(now_ch, now_ch, self.tdim, dropout, attn=False),
        ])

        # ---------- Decoder：逐级上采样，并和 encoder 的 skip 拼接 ----------
        self.upblocks = nn.ModuleList()
        for i, mult in reversed(list(enumerate(ch_mult))):
            out_channels = ch * mult
            for _ in range(num_res_blocks + 1):
                skip_ch = chs.pop()
                self.upblocks.append(ResBlock(now_ch + skip_ch, out_channels, self.tdim, dropout, attn=(i in attn)))
                now_ch = out_channels

            if i != 0:
                self.upblocks.append(UpSample(now_ch))

        assert len(chs) == 0

        self.tail = nn.Sequential(
            group_norm(now_ch),
            Swish(),
            nn.Conv2d(now_ch, out_ch, 3, stride=1, padding=1),
        )
        self.initialize()

    def initialize(self):
        init.xavier_uniform_(self.head.weight)
        init.zeros_(self.head.bias)
        init.xavier_uniform_(self.tail[-1].weight, gain=1e-5)
        init.zeros_(self.tail[-1].bias)

    def forward(self, x, t):
        # x: [B, C, H, W]，t: [B]
        temb = self.time_embedding(t)

        h = self.head(x)
        hs = [h]

        # Encoder：ResBlock 需要时间嵌入；DownSample 不需要。
        for layer in self.downblocks:
            if isinstance(layer, ResBlock):
                h = layer(h, temb)
            else:
                h = layer(h)
            hs.append(h)

        # Middle blocks。
        for layer in self.middleblocks:
            h = layer(h, temb)

        # Decoder：每个 ResBlock 前先把对应 skip connection 拼到通道维。
        for layer in self.upblocks:
            if isinstance(layer, ResBlock):
                skip = hs.pop()

                # 理论上尺寸应该一致；为了兼容奇数尺寸，保险起见对齐一下宽高。
                if h.shape[-2:] != skip.shape[-2:]:
                    h = F.interpolate(h, size=skip.shape[-2:], mode='nearest')

                h = torch.cat([h, skip], dim=1)
                h = layer(h, temb)
            else:
                h = layer(h)

        assert len(hs) == 0
        return self.tail(h)


## 4. DDPM 训练器

训练时不需要真的一步步加噪。DDPM 有闭式形式：

`x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon`

网络目标就是从 `(x_t, t)` 预测出这次加进去的噪声 `epsilon`。

In [ ]:
class GaussianDiffusionTrainer(nn.Module):
    def __init__(self, model, beta_1=1e-4, beta_T=0.02, T=1000):
        super().__init__()
        self.model = model
        self.T = T

        # beta_t 是每一步加入噪声的强度，线性从 beta_1 增加到 beta_T。
        self.register_buffer('betas', torch.linspace(beta_1, beta_T, T).float())
        alphas = 1.0 - self.betas
        alphas_bar = torch.cumprod(alphas, dim=0)

        # 训练公式里需要 sqrt(alpha_bar_t) 和 sqrt(1 - alpha_bar_t)。
        self.register_buffer('sqrt_alphas_bar', torch.sqrt(alphas_bar))
        self.register_buffer('sqrt_one_minus_alphas_bar', torch.sqrt(1.0 - alphas_bar))

    def forward(self, x_0):
        """
        输入干净图片 x_0，返回一个标量 loss。

        注意：x_0 最好已经归一化到 [-1, 1]，因为采样阶段最后也会 clip 到 [-1, 1]。
        """
        B = x_0.shape[0]

        # 每张图片随机抽一个时间步；这样一次 batch 同时训练不同噪声强度。
        t = torch.randint(self.T, size=(B,), device=x_0.device)

        # 真实噪声 epsilon，网络要学会把它预测出来。
        noise = torch.randn_like(x_0)

        # 根据闭式公式直接得到 x_t。
        x_t = (
            extract(self.sqrt_alphas_bar, t, x_0.shape) * x_0 +
            extract(self.sqrt_one_minus_alphas_bar, t, x_0.shape) * noise
        )

        # DDPM 常用 simple loss：预测噪声和真实噪声之间的 MSE。
        pred_noise = self.model(x_t, t)
        loss = F.mse_loss(pred_noise, noise, reduction='mean')
        return loss


## 5. DDPM 采样器

采样时从 `x_T ~ N(0, I)` 开始，按 `T-1 -> ... -> 0` 逐步去噪：

`x_{t-1} = 1/sqrt(alpha_t) * (x_t - (1-alpha_t)/sqrt(1-alpha_bar_t) * epsilon_theta(x_t, t)) + sigma_t * z`

当 `t = 0` 时不再加随机噪声。

In [ ]:
class GaussianDiffusionSampler(nn.Module):
    def __init__(self, model, beta_1=1e-4, beta_T=0.02, T=1000):
        super().__init__()
        self.model = model
        self.T = T

        self.register_buffer('betas', torch.linspace(beta_1, beta_T, T).float())
        alphas = 1.0 - self.betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        alphas_bar_prev = F.pad(alphas_bar, [1, 0], value=1.0)[:T]

        # 反推均值公式中的两个系数。
        self.register_buffer('coeff1', torch.sqrt(1.0 / alphas))
        self.register_buffer('coeff2', self.coeff1 * (1.0 - alphas) / torch.sqrt(1.0 - alphas_bar))

        # 后验方差：beta_t * (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t)。
        self.register_buffer('posterior_var', self.betas * (1.0 - alphas_bar_prev) / (1.0 - alphas_bar))

    def predict_xt_prev_mean_from_eps(self, x_t, t, eps):
        """根据网络预测的噪声 eps，计算 p_theta(x_{t-1} | x_t) 的均值。"""
        assert x_t.shape == eps.shape
        return (
            extract(self.coeff1, t, x_t.shape) * x_t -
            extract(self.coeff2, t, x_t.shape) * eps
        )

    def p_mean_variance(self, x_t, t):
        """返回当前时间步的反向分布均值和方差。"""
        # t=0 时 posterior_var 为 0；为了数值稳定，方差表第 0 项用第 1 项替代。
        var = torch.cat([self.posterior_var[1:2], self.betas[1:]])
        var = extract(var, t, x_t.shape)

        eps = self.model(x_t, t)
        xt_prev_mean = self.predict_xt_prev_mean_from_eps(x_t, t, eps=eps)
        return xt_prev_mean, var

    @torch.no_grad()
    def forward(self, x_T):
        """
        输入纯噪声 x_T，输出生成图片 x_0。

        x_T 的 shape 由你决定，例如 [16, 1, 28, 28]。
        """
        x_t = x_T
        for time_step in reversed(range(self.T)):
            t = x_t.new_full((x_t.shape[0],), time_step, dtype=torch.long)
            mean, var = self.p_mean_variance(x_t=x_t, t=t)

            # Algorithm 2：t > 0 时加 z，t = 0 时 z = 0。
            if time_step > 0:
                noise = torch.randn_like(x_t)
            else:
                noise = 0

            x_t = mean + torch.sqrt(var) * noise
            assert torch.isnan(x_t).int().sum() == 0, '采样过程中出现 NaN'

        x_0 = x_t
        return torch.clip(x_0, -1.0, 1.0)


## 6. 冒烟测试：先检查维度能不能跑通

训练前先跑一个很小的 batch，确认 UNet、训练器、采样器的输入输出 shape 都正确。

In [ ]:
# 冒烟测试只验证结构是否跑通，所以用更小的模型和更少的扩散步，CPU 上也能较快完成。
# 正式训练代码在下一节，仍然使用 T=1000、ch=64。
T = 100
model = UNet(T=T, ch=16, ch_mult=(1, 2, 2), attn=(1,), num_res_blocks=1, dropout=0.1, in_ch=1, out_ch=1).to(device)
trainer = GaussianDiffusionTrainer(model, beta_1=1e-4, beta_T=0.02, T=T).to(device)
sampler = GaussianDiffusionSampler(model, beta_1=1e-4, beta_T=0.02, T=T).to(device)

# 随机图片模拟一个 batch。MNIST 是 [B, 1, 28, 28]。
x = torch.randn(2, 1, 28, 28, device=device)
t = torch.randint(T, (2,), device=device)

with torch.no_grad():
    pred_noise = model(x, t)
    loss = trainer(x)

print('UNet 输出 shape:', pred_noise.shape)
print('训练 loss:', float(loss))


## 7. MNIST 训练代码

下面是完整训练入口。为了 notebook 第一次运行不至于太慢，默认只训练 `epochs = 1`。想要更好效果，可以增加 `epochs`，或者把 `T` 降到 200 先快速验证流程。

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


# DDPM 常用 [-1, 1] 归一化：ToTensor 得到 [0, 1]，Normalize((0.5,), (0.5,)) 后变成 [-1, 1]。
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

data_dir = Path('./data')
train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)

# 如果只是想先快速跑通，可以打开下面这一行，只用 4096 张图片训练。
# train_dataset = Subset(train_dataset, range(4096))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0, pin_memory=(device.type == 'cuda'))

# 如果显存/速度吃紧，可以把 ch 从 64 改成 32，或者把 T 从 1000 改成 200。
T = 1000
model = UNet(T=T, ch=64, ch_mult=(1, 2, 2), attn=(1,), num_res_blocks=2, dropout=0.1, in_ch=1, out_ch=1).to(device)
trainer = GaussianDiffusionTrainer(model, beta_1=1e-4, beta_T=0.02, T=T).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

epochs = 1
model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    total_images = 0

    for step, (images, _) in enumerate(train_loader, start=1):
        images = images.to(device)

        optimizer.zero_grad()
        loss = trainer(images)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_images += images.size(0)

        if step % 100 == 0:
            print(f'Epoch {epoch:02d} | step {step:04d} | loss={loss.item():.4f}')

    print(f'Epoch {epoch:02d} 完成 | avg_loss={total_loss / total_images:.4f}')

torch.save(model.state_dict(), 'ddpm_mnist.pth')
print('模型权重已保存到 ddpm_mnist.pth')


## 8. 采样生成图片

训练完成后，从标准正态噪声开始反推。刚训练 1 个 epoch 的结果通常还会比较糊，训练越久数字越清楚。

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid


# 如果你已经保存过权重，可以取消注释这两行直接加载。
# model.load_state_dict(torch.load('ddpm_mnist.pth', map_location=device))
# model.to(device)

model.eval()
sampler = GaussianDiffusionSampler(model, beta_1=1e-4, beta_T=0.02, T=T).to(device)

# 从纯噪声开始生成 16 张 28x28 灰度图。
x_T = torch.randn(16, 1, 28, 28, device=device)
samples = sampler(x_T)

# [-1, 1] -> [0, 1]，方便 matplotlib 显示。
samples = (samples + 1) / 2
grid = make_grid(samples, nrow=4).permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(5, 5))
plt.imshow(grid, cmap='gray')
plt.axis('off')
plt.show()
